# Ablation RO-A — initial condition only

Solid-body rotation about the domain centre. The exact solution is a circle
rotating rigidly, available in closed form, and it remains a signed distance
function for all time -- so the eikonal constraint is unambiguous here.

Velocity fixed; only the initial interface varies. This is the standard
single-input-function protocol in the operator literature, so the numbers are
comparable to it.

Three arms x three seeds, 16 training instances.

| | |
|---|---|
| grid | 64 x 64 x 32 |
| test set | 100 held-out instances |
| Adam | 30,000 steps, batch 4 |
| model | width 20, modes (12,12,8) |
| eikonal weight | SAW, zero-seeded |
| seeds | 42, 43, 44 |

No L-BFGS stage: adding a refinement that helps one arm more than another
would confound the comparison.

The budget is 30,000 iterations here against 20,000 for the reversed vortex.
That is set by convergence, not by difficulty. The vortex drives its eikonal
weight to order 1e-4, leaving a single-objective loss that settles quickly;
rotation keeps the constraint active throughout and trains against two terms
whose parameter gradients are measurably opposed. A companion run of Part A at
20,000 iterations is retained as a convergence ablation.

The eikonal weight uses `--saw_init zero`. Under a hard initial condition the
field starts exactly at phi_0, which IS a signed distance function, so g_eik
starts near zero, the raw ratio saturates, and the published seeding pins the
weight at its clamp for thousands of steps. Seeding at zero lets it climb as
the field genuinely departs from phi_0.

## Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
%cd /content/drive/MyDrive/NORO
!ls *.py

/content/drive/MyDrive/NORO
collect_results.py  fno3d.py	  ro_family.py	     verify_residual.py
eval_checkpoint.py  residuals.py  train_operator.py  visualize.py


### Sanity check

The exact solution should drive the transport residual toward zero under mesh
refinement while a field violating transport stays flat, with the ratio growing
as the mesh refines.

In [3]:
!python verify_residual.py

         h          exact         frozen      ratio
    0.0625      5.032e-03      1.483e-01       29.5
    0.0312      1.529e-03      1.510e-01       98.7  (x3.3)
    0.0156      4.471e-04      1.519e-01      339.7  (x3.4)
    0.0078      1.267e-04      1.522e-01     1201.8  (x3.5)


---
## Training

### data-free (physics only)

In [4]:
!python train_operator.py --loss strong --saw --steps 30000 --n_train 16 --n_test 100 --seed 42

device=cuda  grid=64x64x32  loss=strong
benchmark: RO  (velocity fixed)
SAW: q=0.95, beta=0.999, init=ratio
labels: 0/16 instances (data-free)  lambda_data=1
IC: hard (structural)
params=7.38M  train=16  test=100
   250 loss 4.395e-03 | train  39.326% | test  39.300% | mass  47.13% | drift  48.58% | w_eik 9.418e-01 | 0.6m
   500 loss 3.437e-03 | train  39.326% | test  39.290% | mass  50.02% | drift  51.56% | w_eik 8.469e-01 | 1.1m
   750 loss 2.642e-03 | train  39.442% | test  39.397% | mass  55.31% | drift  57.04% | w_eik 8.574e-01 | 1.7m
  1000 loss 2.735e-03 | train  39.508% | test  39.473% | mass  56.11% | drift  57.86% | w_eik 8.745e-01 | 2.2m
  1250 loss 2.626e-03 | train  39.798% | test  39.765% | mass  65.64% | drift  67.71% | w_eik 8.831e-01 | 2.8m
  1500 loss 2.178e-03 | train  39.909% | test  39.866% | mass  61.11% | drift  63.03% | w_eik 8.724e-01 | 3.4m
  1750 loss 2.293e-03 | train  40.196% | test  40.154% | mass  65.21% | drift  67.27% | w_eik 8.126e-01 | 3.9m
  2000 los

In [5]:
!python train_operator.py --loss strong --saw --soft_ic --w_ic 10 --steps 30000 --n_train 16 --n_test 100 --seed 42

device=cuda  grid=64x64x32  loss=strong
benchmark: RO  (velocity fixed)
SAW: q=0.95, beta=0.999, init=ratio
labels: 0/16 instances (data-free)  lambda_data=1
IC: soft (w_ic=10)
params=7.38M  train=16  test=100
   250 loss 2.156e-02 | train  40.329% | test  40.398% | mass  91.93% | drift  94.11% | ic 6.76e-04 | w_eik 9.923e-01 | 0.6m
   500 loss 7.969e-03 | train  44.906% | test  44.913% | mass  86.03% | drift  88.11% | ic 1.13e-04 | w_eik 9.800e-01 | 1.1m
   750 loss 4.399e-03 | train  40.909% | test  40.915% | mass  71.17% | drift  73.80% | ic 3.53e-05 | w_eik 9.833e-01 | 1.7m
  1000 loss 4.501e-03 | train  40.199% | test  40.215% | mass  62.94% | drift  63.52% | ic 1.02e-04 | w_eik 9.864e-01 | 2.2m
  1250 loss 3.357e-03 | train  40.045% | test  40.058% | mass  62.20% | drift  61.14% | ic 1.60e-05 | w_eik 9.894e-01 | 2.8m
  1500 loss 2.924e-03 | train  39.994% | test  40.000% | mass  58.57% | drift  56.68% | ic 4.22e-05 | w_eik 9.917e-01 | 3.4m
  1750 loss 2.498e-03 | train  39.977% |

In [6]:
!python train_operator.py --loss strong --w_eik 0 --steps 30000 --n_train 16 --n_test 100 --seed 42

device=cuda  grid=64x64x32  loss=strong
benchmark: RO  (velocity fixed)
labels: 0/16 instances (data-free)  lambda_data=1
IC: hard (structural)
params=7.38M  train=16  test=100
   250 loss 2.678e-03 | train  38.920% | test  38.896% | mass  94.62% | drift  97.67% | 0.6m
   500 loss 2.261e-03 | train  38.495% | test  38.483% | mass  96.13% | drift  99.23% | 1.1m
   750 loss 1.474e-03 | train  38.131% | test  38.130% | mass  96.64% | drift  99.76% | 1.6m
  1000 loss 9.386e-04 | train  31.408% | test  31.506% | mass  94.65% | drift  97.70% | 2.2m
  1250 loss 1.152e-03 | train  27.894% | test  28.014% | mass  90.64% | drift  93.55% | 2.7m
  1500 loss 3.932e-04 | train  24.162% | test  24.263% | mass  59.80% | drift  61.71% | 3.3m
  1750 loss 3.878e-04 | train  23.545% | test  23.630% | mass  55.11% | drift  56.86% | 3.8m
  2000 loss 2.169e-04 | train  22.490% | test  22.558% | mass  36.26% | drift  37.39% | 4.4m
  2250 loss 2.266e-04 | train  22.126% | test  22.181% | mass  36.22% | drift  

In [ ]:
!python train_operator.py --loss strong --saw --saw_init zero --steps 30000 --n_train 16 --n_test 100 --seed 42

device=cuda  grid=64x64x32  loss=strong
benchmark: RO  (velocity fixed)
SAW: q=0.95, beta=0.999, init=zero
labels: 0/16 instances (data-free)  lambda_data=1
IC: hard (structural)
params=7.38M  train=16  test=100
   250 loss 3.184e-03 | train  39.275% | test  39.245% | mass  50.51% | drift  52.07% | w_eik 7.165e-02 | 0.6m
   500 loss 2.987e-03 | train  39.328% | test  39.302% | mass  55.72% | drift  57.45% | w_eik 2.389e-01 | 1.1m
   750 loss 1.957e-03 | train  39.768% | test  39.746% | mass  65.10% | drift  67.15% | w_eik 3.682e-01 | 1.6m
  1000 loss 1.837e-03 | train  40.086% | test  40.073% | mass  64.94% | drift  66.99% | w_eik 4.113e-01 | 2.2m
  1250 loss 1.551e-03 | train  39.991% | test  39.978% | mass  64.56% | drift  66.60% | w_eik 3.814e-01 | 2.7m
  1500 loss 1.298e-03 | train  39.106% | test  39.084% | mass  60.06% | drift  61.94% | w_eik 3.282e-01 | 3.3m
  1750 loss 1.205e-03 | train  38.670% | test  38.657% | mass  58.19% | drift  60.01% | w_eik 2.766e-01 | 3.8m
  2000 loss

In [ ]:
!python train_operator.py --loss strong --saw --saw_init zero --steps 30000 --n_train 16 --n_test 100 --seed 43

device=cuda  grid=64x64x32  loss=strong
benchmark: RO  (velocity fixed)
SAW: q=0.95, beta=0.999, init=zero
labels: 0/16 instances (data-free)  lambda_data=1
IC: hard (structural)
params=7.38M  train=16  test=100
   250 loss 3.069e-03 | train  39.492% | test  39.596% | mass  59.48% | drift  61.45% | w_eik 6.365e-02 | 0.6m
   500 loss 2.575e-03 | train  39.294% | test  39.399% | mass  57.72% | drift  59.63% | w_eik 2.146e-01 | 1.1m
   750 loss 1.874e-03 | train  39.831% | test  39.951% | mass  66.40% | drift  68.58% | w_eik 3.371e-01 | 1.7m
  1000 loss 1.526e-03 | train  40.102% | test  40.221% | mass  66.84% | drift  69.03% | w_eik 3.835e-01 | 2.2m
  1250 loss 1.614e-03 | train  40.361% | test  40.490% | mass  69.64% | drift  71.92% | w_eik 3.938e-01 | 2.8m
  1500 loss 1.326e-03 | train  40.059% | test  40.183% | mass  65.50% | drift  67.65% | w_eik 3.634e-01 | 3.3m
  1750 loss 1.171e-03 | train  39.588% | test  39.709% | mass  62.08% | drift  64.13% | w_eik 3.164e-01 | 3.9m
  2000 loss

In [ ]:
!python train_operator.py --loss strong --saw --saw_init zero --steps 30000 --n_train 16 --n_test 100 --seed 44

device=cuda  grid=64x64x32  loss=strong
benchmark: RO  (velocity fixed)
SAW: q=0.95, beta=0.999, init=zero
labels: 0/16 instances (data-free)  lambda_data=1
IC: hard (structural)
params=7.38M  train=16  test=100
   250 loss 3.207e-03 | train  39.313% | test  39.290% | mass  50.49% | drift  52.10% | w_eik 5.823e-02 | 0.6m
   500 loss 2.457e-03 | train  39.538% | test  39.524% | mass  61.67% | drift  63.64% | w_eik 1.436e-01 | 1.1m
   750 loss 1.638e-03 | train  39.815% | test  39.808% | mass  65.17% | drift  67.25% | w_eik 1.434e-01 | 1.7m
  1000 loss 1.482e-03 | train  39.962% | test  39.961% | mass  66.20% | drift  68.31% | w_eik 1.414e-01 | 2.3m
  1250 loss 1.281e-03 | train  38.956% | test  38.960% | mass  62.59% | drift  64.58% | w_eik 1.275e-01 | 2.8m
  1500 loss 1.005e-03 | train  37.685% | test  37.698% | mass  58.92% | drift  60.80% | w_eik 1.099e-01 | 3.4m
  1750 loss 7.642e-04 | train  36.278% | test  36.293% | mass  59.59% | drift  61.49% | w_eik 9.041e-02 | 3.9m
  2000 loss

In [ ]:
!python collect_results.py

arm         v        n       grid  steps sd    train     test     mass    min
-----------------------------------------------------------------------------
hybrid-8    fixed   16    64^2x32  20000 42   1.184%   1.845%    4.10%   44.5
hybrid-8    fixed   16    64^2x32  20000 43   1.077%   1.875%    4.04%   44.8
hybrid-8    fixed   16    64^2x32  20000 44   0.938%   1.656%    3.38%   44.6
data-free   fixed   16    64^2x32  20000 42   4.930%   5.258%    5.50%   45.0
data-free   fixed   16    64^2x32  20000 43   2.956%   3.258%    4.78%   45.0
data-free   fixed   16    64^2x32  20000 44   5.254%   5.632%    4.03%   43.7
data-free   fixed   16    64^2x32  30000 42   3.739%   4.042%    3.71%   66.0
data-free   fixed   16    64^2x32  30000 43   2.331%   2.630%    2.87%   67.3
data-free   fixed   16    64^2x32  30000 44   4.368%   4.741%    2.86%   67.5
supervised  fixed   16    64^2x32  20000 42   0.402%   2.441%    7.97%   41.7
supervised  fixed   16    64^2x32  20000 43   0.429%   2.611%   

### supervised (labels only)

In [ ]:
!python train_operator.py --loss supervised --steps 30000 --n_train 16 --n_test 100 --seed 42

device=cuda  grid=64x64x32  loss=supervised
benchmark: RO  (velocity fixed)
IC: hard (structural)
params=7.38M  train=16  test=100
   250 loss 8.356e-03 | train  26.782% | test  27.193% | mass  67.28% | drift  69.43% | 0.5m
   500 loss 2.186e-03 | train  12.763% | test  13.754% | mass  28.75% | drift  29.66% | 1.0m
   750 loss 4.965e-04 | train   6.239% | test   7.521% | mass  17.01% | drift  17.56% | 1.6m
  1000 loss 3.113e-04 | train   4.516% | test   5.982% | mass  14.28% | drift  14.76% | 2.1m
  1250 loss 1.755e-04 | train   3.680% | test   5.193% | mass  12.62% | drift  13.01% | 2.7m
  1500 loss 1.243e-04 | train   3.063% | test   4.594% | mass  12.28% | drift  12.70% | 3.2m
  1750 loss 1.204e-04 | train   2.802% | test   4.374% | mass  11.58% | drift  11.94% | 3.7m
  2000 loss 5.756e-05 | train   2.187% | test   3.870% | mass  11.09% | drift  11.46% | 4.3m
  2250 loss 5.104e-05 | train   1.983% | test   3.709% | mass  10.43% | drift  10.73% | 4.8m
  2500 loss 6.331e-05 | train   

In [ ]:
!python train_operator.py --loss supervised --steps 30000 --n_train 16 --n_test 100 --seed 43

device=cuda  grid=64x64x32  loss=supervised
benchmark: RO  (velocity fixed)
IC: hard (structural)
params=7.38M  train=16  test=100
   250 loss 8.777e-03 | train  26.847% | test  27.212% | mass  70.15% | drift  72.43% | 0.5m
   500 loss 2.518e-03 | train  13.470% | test  15.179% | mass  39.53% | drift  40.80% | 1.0m
   750 loss 5.140e-04 | train   6.302% | test   8.145% | mass  22.49% | drift  23.22% | 1.6m
  1000 loss 3.030e-04 | train   4.770% | test   6.768% | mass  18.70% | drift  19.37% | 2.1m
  1250 loss 2.367e-04 | train   4.020% | test   6.024% | mass  16.24% | drift  16.81% | 2.6m
  1500 loss 1.637e-04 | train   3.414% | test   5.412% | mass  14.86% | drift  15.39% | 3.1m
  1750 loss 9.794e-05 | train   2.879% | test   4.939% | mass  13.62% | drift  14.12% | 3.7m
  2000 loss 8.030e-05 | train   2.465% | test   4.527% | mass  13.08% | drift  13.56% | 4.2m
  2250 loss 5.408e-05 | train   2.159% | test   4.184% | mass  12.69% | drift  13.14% | 4.7m
  2500 loss 4.326e-05 | train   

In [ ]:
!python train_operator.py --loss supervised --steps 30000 --n_train 16 --n_test 100 --seed 44

device=cuda  grid=64x64x32  loss=supervised
benchmark: RO  (velocity fixed)
IC: hard (structural)
params=7.38M  train=16  test=100
   250 loss 1.003e-02 | train  26.716% | test  26.827% | mass  71.55% | drift  73.86% | 0.5m
   500 loss 1.559e-03 | train  10.438% | test  11.439% | mass  23.51% | drift  24.28% | 1.0m
   750 loss 3.537e-04 | train   5.684% | test   7.098% | mass  14.77% | drift  15.23% | 1.6m
  1000 loss 2.834e-04 | train   4.365% | test   6.047% | mass  14.22% | drift  14.67% | 2.1m
  1250 loss 1.771e-04 | train   3.486% | test   5.258% | mass  14.09% | drift  14.56% | 2.6m
  1500 loss 1.127e-04 | train   2.832% | test   4.751% | mass  14.08% | drift  14.54% | 3.1m
  1750 loss 7.577e-05 | train   2.331% | test   4.371% | mass  13.19% | drift  13.61% | 3.6m
  2000 loss 6.887e-05 | train   2.107% | test   4.096% | mass  13.30% | drift  13.71% | 4.2m
  2250 loss 3.886e-05 | train   1.864% | test   3.962% | mass  12.74% | drift  13.13% | 4.7m
  2500 loss 3.287e-05 | train   

In [ ]:
!python collect_results.py

arm         v        n       grid  steps sd    train     test     mass    min
-----------------------------------------------------------------------------
hybrid-8    fixed   16    64^2x32  20000 42   1.184%   1.845%    4.10%   44.5
hybrid-8    fixed   16    64^2x32  20000 43   1.077%   1.875%    4.04%   44.8
hybrid-8    fixed   16    64^2x32  20000 44   0.938%   1.656%    3.38%   44.6
data-free   fixed   16    64^2x32  20000 42   4.930%   5.258%    5.50%   45.0
data-free   fixed   16    64^2x32  20000 43   2.956%   3.258%    4.78%   45.0
data-free   fixed   16    64^2x32  20000 44   5.254%   5.632%    4.03%   43.7
data-free   fixed   16    64^2x32  30000 42   3.739%   4.042%    3.71%   66.0
data-free   fixed   16    64^2x32  30000 43   2.331%   2.630%    2.87%   67.3
data-free   fixed   16    64^2x32  30000 44   4.368%   4.741%    2.86%   67.5
supervised  fixed   16    64^2x32  20000 42   0.402%   2.441%    7.97%   41.7
supervised  fixed   16    64^2x32  20000 43   0.429%   2.611%   

### hybrid (8 of 16 labelled)

In [ ]:
!python train_operator.py --loss strong --saw --saw_init zero --n_labelled 8 --steps 30000 --n_train 16 --n_test 100 --seed 42

device=cuda  grid=64x64x32  loss=strong
benchmark: RO  (velocity fixed)
SAW: q=0.95, beta=0.999, init=zero
labels: 8/16 instances  lambda_data=1
IC: hard (structural)
params=7.38M  train=16  test=100
   250 loss 1.953e-02 | train  33.919% | test  33.995% | mass  51.25% | drift  52.84% | data 1.46e-02 | w_eik 1.462e-01 | 0.6m
   500 loss 1.760e-02 | train  33.152% | test  33.266% | mass  46.35% | drift  47.77% | data 1.23e-02 | w_eik 3.350e-01 | 1.1m
   750 loss 1.484e-02 | train  30.277% | test  30.603% | mass  50.76% | drift  52.34% | data 8.38e-03 | w_eik 4.791e-01 | 1.7m
  1000 loss 1.201e-02 | train  24.443% | test  24.989% | mass  38.65% | drift  39.81% | data 5.70e-03 | w_eik 4.834e-01 | 2.2m
  1250 loss 7.626e-03 | train  17.721% | test  18.418% | mass  32.95% | drift  33.93% | data 2.13e-03 | w_eik 4.474e-01 | 2.8m
  1500 loss 3.956e-03 | train  14.490% | test  15.240% | mass  24.58% | drift  25.28% | data 9.70e-04 | w_eik 3.956e-01 | 3.4m
  1750 loss 2.977e-03 | train  12.111%

In [ ]:
!python train_operator.py --loss strong --saw --saw_init zero --n_labelled 8 --steps 30000 --n_train 16 --n_test 100 --seed 43

device=cuda  grid=64x64x32  loss=strong
benchmark: RO  (velocity fixed)
SAW: q=0.95, beta=0.999, init=zero
labels: 8/16 instances  lambda_data=1
IC: hard (structural)
params=7.38M  train=16  test=100
   250 loss 1.638e-02 | train  33.851% | test  34.178% | mass  63.06% | drift  65.14% | data 1.14e-02 | w_eik 1.581e-01 | 0.6m
   500 loss 2.032e-02 | train  33.944% | test  34.383% | mass  50.54% | drift  52.24% | data 1.48e-02 | w_eik 3.439e-01 | 1.1m
   750 loss 1.767e-02 | train  33.324% | test  33.913% | mass  43.48% | drift  44.95% | data 1.31e-02 | w_eik 4.878e-01 | 1.7m
  1000 loss 1.033e-02 | train  27.339% | test  28.235% | mass  44.74% | drift  46.27% | data 5.40e-03 | w_eik 5.427e-01 | 2.3m
  1250 loss 7.880e-03 | train  23.636% | test  24.805% | mass  39.96% | drift  41.35% | data 2.65e-03 | w_eik 5.107e-01 | 2.8m
  1500 loss 5.212e-03 | train  17.915% | test  19.371% | mass  36.75% | drift  38.05% | data 1.64e-03 | w_eik 4.384e-01 | 3.4m
  1750 loss 2.423e-03 | train  12.547%

In [ ]:
!python train_operator.py --loss strong --saw --saw_init zero --n_labelled 8 --steps 30000 --n_train 16 --n_test 100 --seed 44

device=cuda  grid=64x64x32  loss=strong
benchmark: RO  (velocity fixed)
SAW: q=0.95, beta=0.999, init=zero
labels: 8/16 instances  lambda_data=1
IC: hard (structural)
params=7.38M  train=16  test=100
   250 loss 1.642e-02 | train  33.977% | test  34.042% | mass  46.00% | drift  47.46% | data 1.06e-02 | w_eik 1.441e-01 | 0.6m
   500 loss 1.630e-02 | train  33.466% | test  33.560% | mass  46.90% | drift  48.39% | data 1.12e-02 | w_eik 3.326e-01 | 1.1m
   750 loss 1.987e-02 | train  33.363% | test  33.494% | mass  47.76% | drift  49.28% | data 1.52e-02 | w_eik 4.799e-01 | 1.7m
  1000 loss 1.980e-02 | train  27.960% | test  28.192% | mass  45.21% | drift  46.64% | data 1.40e-02 | w_eik 5.608e-01 | 2.3m
  1250 loss 1.438e-02 | train  21.748% | test  22.060% | mass  37.26% | drift  38.44% | data 8.71e-03 | w_eik 5.248e-01 | 2.8m
  1500 loss 8.778e-03 | train  16.811% | test  17.240% | mass  32.03% | drift  33.03% | data 4.77e-03 | w_eik 4.447e-01 | 3.4m
  1750 loss 2.670e-03 | train  11.000%

In [ ]:
!python collect_results.py

arm         v        n       grid  steps sd    train     test     mass    min
-----------------------------------------------------------------------------
hybrid-8    fixed   16    64^2x32  20000 42   1.184%   1.845%    4.10%   44.5
hybrid-8    fixed   16    64^2x32  20000 43   1.077%   1.875%    4.04%   44.8
hybrid-8    fixed   16    64^2x32  20000 44   0.938%   1.656%    3.38%   44.6
hybrid-8    fixed   16    64^2x32  30000 42   0.959%   1.615%    3.72%   67.2
hybrid-8    fixed   16    64^2x32  30000 43   0.897%   1.674%    3.38%   68.0
hybrid-8    fixed   16    64^2x32  30000 44   0.806%   1.508%    3.16%   67.3
data-free   fixed   16    64^2x32  20000 42   4.930%   5.258%    5.50%   45.0
data-free   fixed   16    64^2x32  20000 43   2.956%   3.258%    4.78%   45.0
data-free   fixed   16    64^2x32  20000 44   5.254%   5.632%    4.03%   43.7
data-free   fixed   16    64^2x32  30000 42   3.739%   4.042%    3.71%   66.0
data-free   fixed   16    64^2x32  30000 43   2.331%   2.630%   

---
## Results

In [ ]:
!python collect_results.py --sort test

arm         v        n       grid  steps sd    train     test     mass    min
-----------------------------------------------------------------------------
hybrid-8    fixed   16    64^2x32  30000 44   0.806%   1.508%    3.16%   67.3
hybrid-8    fixed   16    64^2x32  30000 42   0.959%   1.615%    3.72%   67.2
hybrid-8    fixed   16    64^2x32  20000 44   0.938%   1.656%    3.38%   44.6
hybrid-8    fixed   16    64^2x32  30000 43   0.897%   1.674%    3.38%   68.0
hybrid-8    fixed   16    64^2x32  20000 42   1.184%   1.845%    4.10%   44.5
hybrid-8    fixed   16    64^2x32  20000 43   1.077%   1.875%    4.04%   44.8
supervised  fixed   16    64^2x32  30000 42   0.314%   2.426%    7.88%   64.4
supervised  fixed   16    64^2x32  20000 42   0.402%   2.441%    7.97%   41.7
supervised  fixed   16    64^2x32  30000 43   0.331%   2.558%    8.24%   63.7
supervised  fixed   16    64^2x32  20000 43   0.429%   2.611%    8.87%   41.7
data-free   fixed   16    64^2x32  30000 43   2.331%   2.630%   

In [ ]:
!python eval_checkpoint.py strong_saw_saw0_n16_30k_64x32_s42 --n_test 100


strong_saw_saw0_n16_30k_64x32_s42   loss=strong  IC=hard  n_train=16  grid=64^2 x 32  bench=RO  v=fixed  n_test=100
set         rel L2     abs L2   mass(ref)  mass(pi R^2)     drift
-----------------------------------------------------------------
train       3.739%  1.302e-02       2.63%         2.58%     2.94%
test        4.042%  1.405e-02       3.71%         3.65%     3.80%
(exact)     0.000%  0.000e+00       0.00%         0.67%     1.02%


In [ ]:
!python eval_checkpoint.py supervised_n16_30k_64x32_s42 --n_test 100


supervised_n16_30k_64x32_s42   loss=supervised  IC=hard  n_train=16  grid=64^2 x 32  bench=RO  v=fixed  n_test=100
set         rel L2     abs L2   mass(ref)  mass(pi R^2)     drift
-----------------------------------------------------------------
train       0.314%  1.085e-03       0.65%         0.85%     1.13%
test        2.426%  8.524e-03       7.88%         7.83%     8.14%
(exact)     0.000%  0.000e+00       0.00%         0.67%     1.02%


In [ ]:
!python eval_checkpoint.py strong_saw_lab8_saw0_n16_30k_64x32_s42 --n_test 100


strong_saw_lab8_saw0_n16_30k_64x32_s42   loss=strong  IC=hard  n_train=16  grid=64^2 x 32  bench=RO  v=fixed  n_test=100
set         rel L2     abs L2   mass(ref)  mass(pi R^2)     drift
-----------------------------------------------------------------
train       0.959%  3.372e-03       2.16%         2.20%     2.30%
test        1.615%  5.629e-03       3.72%         3.68%     3.84%
(exact)     0.000%  0.000e+00       0.00%         0.67%     1.02%


### Figures

Contours are what matter: a relative L2 number cannot distinguish a slightly
displaced interface from a smeared one.

In [ ]:
!python visualize.py strong_saw_saw0_n16_30k_64x32_s42 supervised_n16_30k_64x32_s42 strong_saw_lab8_saw0_n16_30k_64x32_s42 --n_test 100

plotting instance 38: p=[0.231 1.943 0.109 1.    0.5   0.5  ] omega=1.000
  physics only           mean 4.042%  this instance 3.609%
  supervised             mean 2.426%  this instance 2.514%
  hybrid (8 labels)      mean 1.615%  this instance 2.040%

wrote interfaces_strong_saw_saw0_n16_30k_64x32_s42.png, fields_strong_saw_saw0_n16_30k_64x32_s42.png, spread_strong_saw_saw0_n16_30k_64x32_s42.png


In [ ]:
!python visualize.py strong_saw_saw0_n16_30k_64x32_s42 supervised_n16_30k_64x32_s42 strong_saw_lab8_saw0_n16_30k_64x32_s42 --n_test 100 --instance 50

plotting instance 50: p=[0.254 2.645 0.136 1.    0.5   0.5  ] omega=1.000
  physics only           mean 4.042%  this instance 5.341%
  supervised             mean 2.426%  this instance 10.451%
  hybrid (8 labels)      mean 1.615%  this instance 4.280%

wrote interfaces_strong_saw_saw0_n16_30k_64x32_s42.png, fields_strong_saw_saw0_n16_30k_64x32_s42.png, spread_strong_saw_saw0_n16_30k_64x32_s42.png


In [ ]:
!python visualize.py strong_saw_saw0_n16_30k_64x32_s42 supervised_n16_30k_64x32_s42 strong_saw_lab8_saw0_n16_30k_64x32_s42 --n_test 100 --instance 54

plotting instance 54: p=[0.215 6.029 0.11  1.    0.5   0.5  ] omega=1.000
  physics only           mean 4.042%  this instance 4.145%
  supervised             mean 2.426%  this instance 2.278%
  hybrid (8 labels)      mean 1.615%  this instance 1.566%

wrote interfaces_strong_saw_saw0_n16_30k_64x32_s42.png, fields_strong_saw_saw0_n16_30k_64x32_s42.png, spread_strong_saw_saw0_n16_30k_64x32_s42.png
